# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya
This notebook guides users in exploring and processing the FAIR<sup>2</sup> dataset using the `mlcroissant` library, referencing entities by their `@id` as per Croissant best practices.

### Dataset Source
The dataset is defined by a Croissant schema and available via:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields and columns using `@id` reference.
This allows precise referencing and exploration using `mlcroissant`.

In [ ]:
# List all record sets and their fields referenced by @id
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print('No record sets found in this FAIR2 dataset metadata.')
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']} - {rs.get('name', 'No name')}")
        fields = rs.get('field', [])
        print('Fields:')
        for f in fields:
            print(f"  Field @id: {f['@id']} ({f.get('name', 'No name')})")

# If there are record sets, print some records using their @id
if record_sets:
    # Choose first RecordSet for exploration
    record_set_id = record_sets[0]['@id']
    print(f"\nSample records in RecordSet @id {record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if i > 2:
            break

## 3. Data Extraction
Load all record sets into DataFrames for analysis.

All references are made by the appropriate Croissant `@id`s for record sets and fields.

In [ ]:
# Extract data from each record set
dataframes = {}
record_set_ids = []
for rs in getattr(metadata, 'recordSet', []):
    rs_id = rs['@id']
    record_set_ids.append(rs_id)
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

# Show columns for first available record set
if dataframes:
    first_rs_id = record_set_ids[0]
    print(f"Columns for RecordSet @id {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print("No DataFrames extracted; dataset may be metadata-only or schema-only.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filtering records
- Normalizing numeric fields
- Grouping/categorizing

All operations reference fields by full `@id` as required for Croissant compliance.

In [ ]:
# Example EDA on first record set with numeric and categorical fields
if dataframes:
    df = dataframes[first_rs_id]

    # Try to select a numeric field by @id
    numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
    if not numeric_fields:
        print("No numeric fields found in the first record set.")
    else:
        numeric_field_id = numeric_fields[0]  # Use the first numeric @id
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field
        categorical_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        if categorical_fields:
            group_field_id = categorical_fields[0]
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No categorical group field found for grouping.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize distributions or relationships using `matplotlib`, referencing fields by `@id` for clarity and reproducibility.

In [ ]:
# Example: Histogram and scatterplot
if dataframes:
    df = dataframes[first_rs_id]
    numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        plt.figure(figsize=(7, 4))
        plt.hist(df[numeric_field_id].dropna(), bins=20, color='skyblue', edgecolor='k')
        plt.title(f"Distribution of {numeric_field_id} (by @id)")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()

        # If second numeric, show scatterplot
        if len(numeric_fields) > 1:
            plt.figure(figsize=(7, 5))
            plt.scatter(df[numeric_fields[0]], df[numeric_fields[1]], alpha=0.6)
            plt.title(f"Scatterplot: {numeric_fields[0]} vs {numeric_fields[1]} (@id)")
            plt.xlabel(numeric_fields[0])
            plt.ylabel(numeric_fields[1])
            plt.show()
else:
    print("No visualizable data available.")

## 6. Conclusion
This notebook demonstrates how to load, overview, extract, process, and visualize Croissant dataset entities using the `mlcroissant` library. All dataset elements are referenced via their `@id` for full FAIR<sup>2</sup> compliance. Key findings depend on actual data; please refer to visualizations and outputs above for insights about knowledge adoption predictors in rangeland management in Northern Kenya.